In [158]:
import pandas as pd

In [159]:
df = pd.read_csv("Indian-Railway-Network-and-Delays/train_routes_delays_Sep2024.csv", dtype={"train": str})

In [160]:
df["train"].str.startswith("0").sum()

np.int64(128922)

train column was a int type column so at the time of loading it was treating the values as int values like 0123 will be replace with 123.

We need to fix it at the time of loading. nahi toh we can't fix it later.

In [161]:
df.shape

(1282325, 9)

In [162]:
df.sample(5)

,train,date,station,sch_arr,act_arr,arr_delay,sch_dep,act_dep,dep_delay
600302,16328,2024-09-18,PUK,06:38 AM,06:43 AM,5.0,06:39 AM,06:44 AM,5.0
420370,13152,2024-09-28,BWN,01:38 PM,01:38 PM,0.0,01:41 PM,01:41 PM,0.0
929729,16650,2024-09-07,WKI,12:48 PM,01:28 PM,40.0,12:49 PM,01:29 PM,40.0
83238,14624,2024-09-30,FZR,04:10 AM,04:10 AM,0.0,04:10 AM,04:10 AM,0.0
231191,17480,2024-09-24,VKI,11:49 AM,11:55 AM,6.0,11:50 AM,11:56 AM,6.0


#### Check 1: Are the column types right?

df.dtypes

Expected: train is int64. The time columns are object (text) and the delays are float64.

Why this matters: train is an ID, not a number. Some train numbers start with 0 (like 05290), and reading them as integers turns them into 5290. That affects 745 trains (129k rows), and they will silently fail to match when you join with the routes file.

In [163]:
df.dtypes

train            str
date             str
station          str
sch_arr          str
act_arr          str
arr_delay    float64
sch_dep          str
act_dep          str
dep_delay    float64
dtype: object

#### Are values missing?

Checking for NaN values. IF there is any NaN values it will show.

In [164]:
df.isna().sum()

train        0
date         0
station      0
sch_arr      0
act_arr      0
arr_delay    0
sch_dep      0
act_dep      0
dep_delay    0
dtype: int64

#### Checking that whether any duplicate row exists or not

In [165]:
df.duplicated(['train', 'date', 'station']).sum()

np.int64(0)

No duplicate row exists

In [166]:
round(df[['arr_delay', 'dep_delay']].describe(), 3)

,arr_delay,dep_delay
count,1282325.000,1282325.000
mean,32.897,32.897
std,84.520,84.520
min,0.000,0.000
25%,0.000,0.000
50%,8.000,8.000
75%,28.000,28.000
max,4336.000,4336.000


Median is 8 and mean is 32 which is 4x of Median. It says that the data is skewed.

#### Checking for redundant column

arr_delay and dep_delay are the same column so we can drop any of the column

In [167]:
(df["arr_delay"] == df["dep_delay"]).all()

np.True_

In [168]:
pd.to_datetime("08:05 PM", format="%I:%M %p")

Timestamp('1900-01-01 20:05:00')

Converting the actual time with 24 hrs formating. The Actual arrival and the scheduled arrival.

In [169]:
sch = pd.to_datetime(df["sch_arr"], format="%I:%M %p")
act = pd.to_datetime(df["act_arr"], format="%I:%M %p")
sch.head(3)

0   1900-01-01 08:00:00
1   1900-01-01 09:05:00
2   1900-01-01 09:57:00
Name: sch_arr, dtype: datetime64[us]

In [170]:
act.head(3)

0   1900-01-01 08:01:00
1   1900-01-01 09:20:00
2   1900-01-01 10:00:00
Name: act_arr, dtype: datetime64[us]

Difference between actual and scheduled.

In [171]:
diff = act - sch
diff.head(3)

0   0 days 00:01:00
1   0 days 00:15:00
2   0 days 00:03:00
dtype: timedelta64[us]

Converting it into minutes

In [172]:
calc = diff.dt.total_seconds() / 60
calc.head(3)

0     1.0
1    15.0
2     3.0
dtype: float64

In [173]:
mismatch = calc != df['arr_delay']
mismatch.sum()

np.int64(85442)

Created a local dataset of mismatched rows

In [174]:
check = df.loc[mismatch, ["train", "station", "sch_arr", "act_arr", "arr_delay"]]
check["calc"] = calc[mismatch]
check.head(10)

,train,station,sch_arr,act_arr,arr_delay,calc
0,12303,HWH,08:00 AM,08:01 AM,0.0,1.0
18,12303,DDU,07:30 PM,07:31 PM,0.0,1.0
44,12303,PRYJ,09:40 PM,02:32 AM,292.0,-1148.0
55,12303,JMT,11:15 AM,11:16 AM,0.0,1.0
58,12303,JAJ,01:12 PM,01:13 PM,0.0,1.0
119,12303,PRYJ,09:40 PM,01:08 AM,208.0,-1232.0
125,12303,HWH,08:00 AM,08:01 AM,0.0,1.0
133,12303,JAJ,01:12 PM,01:13 PM,0.0,1.0
143,12303,DDU,07:30 PM,12:28 AM,298.0,-1142.0
144,12303,PRYJ,09:40 PM,02:45 AM,305.0,-1135.0


In [175]:
(check["arr_delay"] - check["calc"]).value_counts()

-1.0       59258
 1440.0    26127
 2880.0       49
 1439.0        4
 4320.0        4
Name: count, dtype: int64

24 * 60 = 1440 min. We are taking modulo so we can address the -ve values as well as the midnight wala issue.

In [176]:
check["calc_fixed"] = calc % 1440
check["calc_fixed"].head(3)

0       1.0
18      1.0
44    292.0
Name: calc_fixed, dtype: float64

In [177]:
check.sample(10)

,train,station,sch_arr,act_arr,arr_delay,calc,calc_fixed
309868,15159,GCT,09:45 AM,09:46 AM,0.0,1.0,1.0
612307,11305,SUR,11:20 AM,11:21 AM,0.0,1.0,1.0
432022,12137,KYN,08:32 PM,01:28 AM,296.0,-1144.0,296.0
480794,12809,IGP,11:42 PM,12:30 AM,48.0,-1392.0,48.0
1211168,12609,MAS,01:35 PM,01:36 PM,0.0,1.0,1.0
674289,14215,PRG,05:54 AM,05:55 AM,0.0,1.0,1.0
214323,09590,PUX,08:47 AM,08:48 AM,0.0,1.0,1.0
927889,11037,MMR,09:45 PM,09:46 PM,0.0,1.0,1.0
259347,22602,SNSI,08:25 AM,08:26 AM,0.0,1.0,1.0
974337,14217,TDL,11:52 PM,12:47 AM,55.0,-1385.0,55.0


If train is 1min delay then we don't consider it as a delay. Marking the value as 0.

In [180]:
check["calc_fixed"] = check["calc_fixed"].replace(1, 0)

In [181]:
check.sample(10)

,train,station,sch_arr,act_arr,arr_delay,calc,calc_fixed
16982,13064,AZ,01:21 AM,01:22 AM,0.0,1.0,0.0
199330,16341,PVU,08:28 AM,08:29 AM,0.0,1.0,0.0
663881,16127,MDU,05:40 PM,05:41 PM,0.0,1.0,0.0
1129669,16032,SSB,12:36 PM,12:37 PM,0.0,1.0,0.0
1214401,22537,BBK,10:03 PM,10:04 PM,0.0,1.0,0.0
393726,11464,UJN,11:50 PM,12:37 AM,47.0,-1393.0,47.0
1159602,14801,AII,12:40 PM,12:41 PM,0.0,1.0,0.0
982561,17310,VSG,10:55 PM,10:56 PM,0.0,1.0,0.0
715705,19812,GWL,08:35 PM,08:36 PM,0.0,1.0,0.0
172766,22954,DRD,01:28 PM,01:29 PM,0.0,1.0,0.0


In [183]:
(check["arr_delay"] == check["calc_fixed"]).sum()

np.int64(85257)

In [184]:
df["date"].value_counts().sort_index()

date
2024-09-01    40571
2024-09-02    40786
2024-09-03    41418
2024-09-04    41738
2024-09-05    41671
2024-09-06    42811
2024-09-07    41616
2024-09-08    42968
2024-09-09    42970
2024-09-10    42055
2024-09-11    41908
2024-09-12    42296
2024-09-13    43334
2024-09-14    42788
2024-09-15    43119
2024-09-16    44604
2024-09-17    44447
2024-09-18    43788
2024-09-19    43432
2024-09-20    43974
2024-09-21    43317
2024-09-22    43172
2024-09-23    43150
2024-09-24    42296
2024-09-25    42315
2024-09-26    43433
2024-09-27    44139
2024-09-28    43033
2024-09-29    42399
2024-09-30    42777
Name: count, dtype: int64

In [185]:
df.groupby("train")["date"].nunique().describe()


count    3892.000000
mean       14.770041
std        11.004067
min         1.000000
25%         4.000000
50%        11.000000
75%        27.000000
max        30.000000
Name: date, dtype: float64

In [188]:
routes = pd.read_csv("Indian-Railway-Network-and-Delays/train_routes_Sep2024.csv", dtype={"trainNumber": str})

routes.head(10)

,stnSerialNumber,trainNumber,trainName,station_code,station_name,distance,arrivalTime,departureTime
0,1,12303,Poorva Express,HWH,Howrah,0,08:00 AM,08:00 AM
1,2,12303,Poorva Express,BWN,Bardhaman,95,09:05 AM,09:08 AM
2,3,12303,Poorva Express,DGR,Durgapur,157,09:57 AM,09:59 AM
3,4,12303,Poorva Express,ASN,Asansol,199,10:32 AM,10:37 AM
4,5,12303,Poorva Express,CRJ,Chittaranjan,224,11:00 AM,11:02 AM
5,6,12303,Poorva Express,JMT,Jamtara,239,11:15 AM,11:17 AM
6,7,12303,Poorva Express,MDP,Madhupur,281,11:44 AM,11:46 AM
7,8,12303,Poorva Express,JSME,Jasidih,310,12:11 PM,12:15 PM
8,9,12303,Poorva Express,JAJ,Jhajha,354,01:12 PM,01:17 PM
9,10,12303,Poorva Express,JMU,Jamui,380,01:33 PM,01:35 PM


Merged the two tables based on the "train", "station" and "trainNumber", "station_code". Left join. 

In [189]:
m = df.merge(routes,
             left_on=["train", "station"],
             right_on=["trainNumber", "station_code"],
             how="left", indicator=True)
m["_merge"].value_counts()

_merge
both          1283333
left_only          62
right_only          0
Name: count, dtype: int64

In [191]:
m.head(5)

,train,date,station,sch_arr,act_arr,arr_delay,sch_dep,act_dep,dep_delay,stnSerialNumber,trainNumber,trainName,station_code,station_name,distance,arrivalTime,departureTime,_merge
0,12303,2024-09-02,HWH,08:00 AM,08:01 AM,0.0,08:00 AM,08:01 AM,0.0,1.0,12303,Poorva Express,HWH,Howrah,0.0,08:00 AM,08:00 AM,both
1,12303,2024-09-02,BWN,09:05 AM,09:20 AM,15.0,09:08 AM,09:23 AM,15.0,2.0,12303,Poorva Express,BWN,Bardhaman,95.0,09:05 AM,09:08 AM,both
2,12303,2024-09-02,DGR,09:57 AM,10:00 AM,3.0,09:59 AM,10:02 AM,3.0,3.0,12303,Poorva Express,DGR,Durgapur,157.0,09:57 AM,09:59 AM,both
3,12303,2024-09-02,ASN,10:32 AM,10:34 AM,2.0,10:37 AM,10:39 AM,2.0,4.0,12303,Poorva Express,ASN,Asansol,199.0,10:32 AM,10:37 AM,both
4,12303,2024-09-02,CRJ,11:00 AM,11:04 AM,4.0,11:02 AM,11:06 AM,4.0,5.0,12303,Poorva Express,CRJ,Chittaranjan,224.0,11:00 AM,11:02 AM,both


42 duplicate value exists.

In [190]:
routes.duplicated(["trainNumber", "station_code"]).sum()


np.int64(42)

In [193]:
df.duplicated(["train", "station"]).sum()

np.int64(1197308)